# UC-CAT-2 — Create Collection with LayerConfig

**Persona:** Data Engineer

**Goal:** Create a physical PostgreSQL-backed STAC collection, apply `PostgresCollectionDriverConfig`
and `RoutingPluginConfig`, verify effective config resolution, and check schema health.
This notebook demonstrates the LayerConfig footgun and how to avoid it.

> **ACTIVE BUG:** Items ingested into a collection that has no `RoutingPluginConfig` will return
> HTTP 201 at ingest time but GET will return 0 results. The platform lacks a driver to route
> the read. This notebook is the canonical fix: always configure routing **before** ingesting items.

**Prerequisites:**
- A catalog already provisioned (run `catalog/01_provision_tenant_catalog.ipynb` first, or
  set `CATALOG_ID` to an existing catalog)
- Admin JWT in `DYNASTORE_ADMIN_TOKEN`
- `driver:postgresql` installed on the platform

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_ADMIN_TOKEN`, `CATALOG_ID`

In [ ]:
import os
import json

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN = os.environ.get("DYNASTORE_ADMIN_TOKEN", "")
CATALOG_ID = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "ndvi_mosaic_2024")

admin_headers = {
    "Authorization": f"Bearer {ADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=admin_headers, timeout=30.0)
print(f"Connected to {BASE_URL}  catalog={CATALOG_ID}  collection={COLLECTION_ID}")

## Step 1 — Create STAC collection

POST a `STACCollectionRequest` to create the physical collection record. At this point
no driver is configured, so no PostgreSQL table exists yet — that is created when the
driver config is applied.

In [ ]:
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": "NDVI 250m Dekadal Mosaic 2024",
    "description": "10-day composite NDVI at 250 m resolution derived from MODIS MOD13Q1.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-18.0, -35.0, 52.0, 38.0]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "links": [],
}

r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
print(r.status_code, r.text[:400])
assert r.status_code == 201, f"Expected 201, got {r.status_code}: {r.text}"

collection = r.json()
assert collection["id"] == COLLECTION_ID
print(f"Collection created: {collection['id']}")

## Step 2 — Configure PostgreSQL driver

Apply `PostgresCollectionDriverConfig` at the collection level. This tells the platform
to create a partitioned table for this collection and which sidecars to activate.

Then apply `RoutingPluginConfig` to instruct the platform which driver to use for each
operation class (WRITE / READ). Without this second step, items will appear to ingest
successfully but will not be retrievable.

In [ ]:
driver_config = {
    "enabled": True,
    "partition_type": "LIST",
    "sidecars": ["item_metadata", "stac_metadata"],
}

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/driver:postgresql",
    content=json.dumps(driver_config),
)
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
print("PostgreSQL driver config applied.")

In [ ]:
routing_config = {
    "enabled": True,
    "operations": {
        "WRITE": [{"driver": "driver:postgresql", "on_failure": "fatal"}],
        "READ": [{"driver": "driver:postgresql", "on_failure": "fatal"}],
    },
}

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/routing",
    content=json.dumps(routing_config),
)
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
print("Routing config applied.")

## Step 3 — Verify effective config resolution

The `/effective` endpoint resolves the config waterfall: collection → catalog → platform → code
defaults. After an explicit PUT at collection level the `source` field must be `"collection"`.

In [ ]:
r = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/driver:postgresql/effective"
)
print(r.status_code, r.text[:500])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

effective = r.json()
# The resolved source must be collection-level since we just PUT it there
source = effective.get("source") or effective.get("resolved", {}).get("source", "")
assert source in ("collection", "catalog", "platform"), (
    f"Unexpected source value: {source!r}"
)
assert source == "collection", (
    f"Expected source 'collection' after explicit PUT, got '{source}'"
)
print(f"Effective config source: '{source}' — correct.")

In [ ]:
# Check schema health — the collection table should appear
r = client.get(f"/admin/schemas/{CATALOG_ID}/health")
print(r.status_code, r.text[:600])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

health = r.json()
# The health payload may be a dict keyed by collection_id or a list
health_str = json.dumps(health)
assert COLLECTION_ID in health_str, (
    f"Collection '{COLLECTION_ID}' not found in schema health response: {health_str[:400]}"
)
print(f"Schema health OK — '{COLLECTION_ID}' is present.")

## Edge Case — Missing RoutingPluginConfig causes silent item loss

> **Root cause #2 of `project_geoid_ingestion_collection_population`:**  
> Ingesting an item into a collection without a `RoutingPluginConfig` returns HTTP 201 (the
> write is accepted by the ingestion gateway) but no driver is wired for READ, so
> subsequent GET returns an empty result set or 404 depending on fallback behaviour.

The cell below demonstrates this with a **scratch collection** that deliberately has no
routing config. The test collection created above is not touched.

In [ ]:
# Create a scratch collection with no routing config to reproduce the bug
SCRATCH_ID = f"{COLLECTION_ID}-scratch-no-routing"

scratch_payload = {
    **collection_payload,
    "id": SCRATCH_ID,
    "title": "Scratch (no routing — bug demo)",
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(scratch_payload),
)
assert r.status_code == 201, f"Expected 201 creating scratch collection, got {r.status_code}: {r.text}"
print(f"Scratch collection created: {SCRATCH_ID}")

# Ingest a minimal item — platform returns 201 because ingestion gateway accepts it
item_payload = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": "test-item-no-routing",
    "geometry": {"type": "Point", "coordinates": [38.5, 8.9]},
    "bbox": [38.5, 8.9, 38.5, 8.9],
    "properties": {"datetime": "2024-01-11T00:00:00Z"},
    "links": [],
    "assets": {},
    "collection": SCRATCH_ID,
}
r_ingest = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{SCRATCH_ID}/items",
    content=json.dumps(item_payload),
)
print(f"Ingest status: {r_ingest.status_code}")
# 201 is expected — the bug is that the item is not actually routed to any storage
# Some platform versions may return 4xx here if no driver is configured at all.

# Attempt to retrieve the item
r_get = client.get(
    f"/stac/catalogs/{CATALOG_ID}/collections/{SCRATCH_ID}/items/test-item-no-routing"
)
print(f"GET item status: {r_get.status_code}  body: {r_get.text[:200]}")
# We assert that the item is NOT found (404) or the collection returns 0 items,
# demonstrating that the 201 at ingest was misleading.
assert r_get.status_code in (404, 200), (
    f"Unexpected status {r_get.status_code}: {r_get.text}"
)
if r_get.status_code == 200:
    body = r_get.json()
    items = body if isinstance(body, list) else body.get("features", body.get("items", []))
    print(
        f"BUG REPRODUCED: ingest returned {r_ingest.status_code} but GET returned "
        f"{len(items)} item(s) — silent item loss confirmed."
    )
else:
    print(
        f"BUG REPRODUCED: ingest returned {r_ingest.status_code} but GET returned 404 "
        "— item was not persisted to any driver."
    )

# Cleanup scratch collection
client.delete(f"/stac/catalogs/{CATALOG_ID}/collections/{SCRATCH_ID}")
print("Scratch collection removed.")

## Teardown

Delete the test collection. The catalog itself is left in place — teardown the catalog
using `catalog/01_provision_tenant_catalog.ipynb` if needed.

In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
print(r.status_code, r.text[:300])
assert r.status_code == 204, f"Expected 204, got {r.status_code}: {r.text}"
print(f"Collection {COLLECTION_ID} deleted.")
client.close()